In [1]:
import polars as pl
import pandas as pd
from tslearn.clustering import TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance 

import warnings
from tqdm import tqdm 
import numpy as np

from kneed import KneeLocator
from scipy.signal import savgol_filter


c:\ProgramData\anaconda3\envs\myConda\Lib\site-packages\tslearn\bases\bases.py:15: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


In [2]:
import httpx

In [3]:
from sklearn.metrics import davies_bouldin_score, silhouette_score

In [4]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

In [5]:
base_uri = 'http://127.0.0.1:8000/api/'

api_routes = {
    1: 'program_university_kpi_latest',
    2: 'program_category',
    3: 'program_noc_link',
    4: 'provincial_noc_group_labor_statistic',
    5: 'noc_grouping',
    6: 'economic_region_employment_estimate',
    7: 'economicregion',
    8: 'province',
    9: 'university',
    10: 'prediction_cache'
}

# Instantiating a synchronous client with 120s timeout
client = httpx.Client(timeout=500.0)

# Generic function for making a request to the API given a formatted URL
def Make_Request(url: str):
    try:
        print(f"URL => {url}")
        response = client.get(url)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Request failed for {url}: {e}")
        return []

def RequestProvince() -> pd.DataFrame:
    try:
        url_to_call = f"{base_uri}{api_routes[8]}/"
        data = Make_Request(url=url_to_call)
        return pd.DataFrame(data)
    except Exception as e:
        print(f"RequestProvince() => {e}")
        return pd.DataFrame()

#region Request Functions

In [6]:
ProvinceData = RequestProvince()

URL => http://127.0.0.1:8000/api/province/


In [7]:
df = pl.read_parquet('Data/SeasonalDecomposedData.parquet')

In [8]:
df.columns

['observed',
 'trend',
 'resid',
 'seasonal_3',
 'seasonal_6',
 'seasonal_12',
 'ds',
 'provinceid',
 'noc_groupingid']

In [9]:
print(df[['provinceid','noc_groupingid','ds','seasonal_3','trend']].head(10))

shape: (10, 5)
┌────────────┬────────────────┬─────────────────────┬────────────┬──────────┐
│ provinceid ┆ noc_groupingid ┆ ds                  ┆ seasonal_3 ┆ trend    │
│ ---        ┆ ---            ┆ ---                 ┆ ---        ┆ ---      │
│ f64        ┆ i64            ┆ datetime[ns]        ┆ f64        ┆ f64      │
╞════════════╪════════════════╪═════════════════════╪════════════╪══════════╡
│ 10.0       ┆ 1              ┆ 2000-01-01 00:00:00 ┆ -0.109472  ┆ 7.378831 │
│ 10.0       ┆ 1              ┆ 2000-02-01 00:00:00 ┆ -0.063244  ┆ 7.410238 │
│ 10.0       ┆ 1              ┆ 2000-03-01 00:00:00 ┆ 0.135241   ┆ 7.440902 │
│ 10.0       ┆ 1              ┆ 2000-04-01 00:00:00 ┆ -0.047459  ┆ 7.470413 │
│ 10.0       ┆ 1              ┆ 2000-05-01 00:00:00 ┆ -0.077763  ┆ 7.497976 │
│ 10.0       ┆ 1              ┆ 2000-06-01 00:00:00 ┆ 0.093296   ┆ 7.522601 │
│ 10.0       ┆ 1              ┆ 2000-07-01 00:00:00 ┆ 0.006749   ┆ 7.543554 │
│ 10.0       ┆ 1              ┆ 2000-08-01 00:00:

In [10]:
ProvinceIDs = df.select(pl.col('provinceid').unique()).to_series().to_list()
print(ProvinceIDs)

[10.0, 11.0, 12.0, 13.0, 24.0, 35.0, 46.0, 47.0, 48.0, 59.0]


In [16]:
def Get_Sample(provinceid):
    try:
    
        Sample = df.filter(pl.col('provinceid')==provinceid).to_pandas()
        Sample['ds'] = pd.to_datetime(Sample['ds'])
        Sample['year'] = Sample['ds'].dt.year
        Sample = Sample[(Sample['year']>= 2020) &(Sample['year'] <= 2024)]
        Sample = Sample.sort_values(by=['noc_groupingid','ds'], ascending=True)

        return Sample
    except Exception as e:
        print(f"Get_Sample(provinceid={provinceid}) => {e}")
        return pd.DataFrame()

def Pivot_Data(Sample: pd.DataFrame, col):
    try:
        Sample_Pivoted = Sample.pivot(index='noc_groupingid',columns='ds',values=col).fillna(0)
        return Sample_Pivoted
    

    except Exception as e:
        print(f"Pivot_Data() => {e}")
        return pd.DataFrame()

def Pre_Process(Sample_Pivoted: pd.DataFrame):
    try:
        # Scale time series (shape-based)
        data_scaled = TimeSeriesScalerMeanVariance().fit_transform(Sample_Pivoted)
        
        # 3D for TimeSeriesKMeans
        data_3d = data_scaled.reshape(data_scaled.shape[0], data_scaled.shape[1], 1)
        
        # 2D for Metrics
        data_2d = data_scaled.reshape(data_scaled.shape[0], -1)

        return data_3d, data_2d

    except Exception as e:
        print(f"Pre_Process() => {e}")
        return None, None

def Get_Metrics_And_K(data_3d, data_2d, upper_k, lower_k=2):
    try:
        interia_list = []
        for k in tqdm(range(lower_k, upper_k)):
            model = TimeSeriesKMeans(n_clusters=k, metric='euclidean',verbose=False)
            labels = model.fit_predict(data_3d)

            

            interia_list.append(
                {
                    'K':k,
                    'Inertia':model.inertia_,
                    'DBI': davies_bouldin_score(data_2d, labels),
                    'Silhouette': silhouette_score(data_2d, labels),
                }
            )

        df = pd.DataFrame(interia_list)

        df['Inertia_Norm'] = (
            (df['Inertia'] - df['Inertia'].min()) /
            (df['Inertia'].max() - df['Inertia'].min())
        )

        df['DBI_Norm'] = (
            (df['DBI'] - df['DBI'].min()) /
            (df['DBI'].max() - df['DBI'].min())
        )

        df['Silhouette_Norm'] = (
            (df['Silhouette'] - df['Silhouette'].min()) /
            (df['Silhouette'].max() - df['Silhouette'].min())
        )

        df['Inertia_Smoothed'] = savgol_filter(
            df['Inertia_Norm'],
            window_length=13,
            polyorder=2
        )

        kl = KneeLocator(
            range(2, upper_k, 1),
            df['Inertia_Smoothed'],
            curve="convex",
            direction="decreasing",
            S=1.0,
            interp_method="polynomial"
            )


        return df, kl.elbow

    except Exception as e:
        print(f"Get_Inertia() => {e}")
        return pd.DataFrame(), 0

def Generate_Cluster_Labels(data_3d, Sample_Pivoted, K, ClusterType="Trend" ):
    try:
        model = TimeSeriesKMeans(n_clusters=K, metric='euclidean',verbose=False)
        labels = model.fit_predict(data_3d)

        Sample_Pivoted[f'{ClusterType} Cluster'] = labels
        Sample_Pivoted_Copy = Sample_Pivoted.reset_index()

        df_Clusters = Sample_Pivoted_Copy.melt(id_vars=['noc_groupingid', f'{ClusterType} Cluster'], var_name='ds', value_name=f'Employment_{ClusterType}')

        df_Cluster_Labels = df_Clusters[['noc_groupingid',f'{ClusterType} Cluster']].drop_duplicates()
        return df_Cluster_Labels

    except Exception as e:
        print(f"Generate_Cluster_Labels() => {e}")
        return None

def Plot_K_Metrics(df_Metrics, K, Province, Variable):
    try:
        
        LinePlot = px.line(
            title=f'K Selection for {Province} - Data Clustered by {Variable} - Suggested K = {K}',
            width=1000,
            height=900,
            template='ggplot2'
        )

        LinePlot.add_traces(
            go.Scatter(
                x=df_Metrics['K'],
                y=df_Metrics['Inertia_Norm'],
                name='Inertia',
                mode='lines',
                line=dict(color='#12488A', width=2.5),
            )
        )

        LinePlot.add_traces(
            go.Scatter(
                x=df_Metrics['K'],
                y=df_Metrics['Inertia_Smoothed'],
                name='Inertia Smoothed',
                mode='lines',
                line=dict(color="#0B2F5B", width=3.5, dash='dash'),
            )
        )

        LinePlot.add_traces(
            go.Scatter(
                x=df_Metrics['K'],
                y=df_Metrics['DBI_Norm'],
                name='DBI',
                mode='lines',
                line=dict(color="#0C533E", width=2.5),
            )
        )

        LinePlot.add_traces(
            go.Scatter(
                x=df_Metrics['K'],
                y=df_Metrics['Silhouette_Norm'],
                name='Silhouette',
                mode='lines',
                line=dict(color='#661C2A', width=2.5),
            )
        )

        if K is not None:
            LinePlot.add_shape(
            type="line",
            x0=K,
            x1=K,
            y0=0,
            y1=1,
            xref="x",
            yref="paper",  
            line=dict(
                color="black",
                width=2,
                dash="dash"
            )
        )

    
            LinePlot.add_annotation(
                x=K,
                y=1,
                xref="x",
                yref="paper",
                text=f"K={K}",
                showarrow=False,
                yanchor="bottom"
            )
        
        LinePlot.update_layout(
            legend=dict(
            orientation="h",      
            yanchor="top",
            y=-0.2,              
            xanchor="center",
            x=0.5                
        )
        )

        return LinePlot



    except Exception as e:
        print(f"Plot_K_Metrics() => {e}")

In [12]:
ProvinceData

,provinceid,province_dguid,province,province_shorthand
0,59,2021A000259,British Columbia,BC
1,24,2021A000224,Quebec,QC
2,11,2021A000211,Prince Edward Island,PE
3,47,2021A000247,Saskatchewan,SK
4,46,2021A000246,Manitoba,MB
5,13,2021A000213,New Brunswick,NB
6,10,2021A000210,Newfoundland and Labrador,NL
7,60,2021A000260,Yukon,YT
8,61,2021A000261,Northwest Territories,NT
9,48,2021A000248,Alberta,AB


In [13]:
ProvinceID_To_ProvinceName_Map = ProvinceData.set_index('provinceid')['province'].to_dict()
ProvinceID_To_ProvinceName_Map

{59: 'British Columbia',
 24: 'Quebec',
 11: 'Prince Edward Island',
 47: 'Saskatchewan',
 46: 'Manitoba',
 13: 'New Brunswick',
 10: 'Newfoundland and Labrador',
 60: 'Yukon',
 61: 'Northwest Territories',
 48: 'Alberta',
 12: 'Nova Scotia',
 62: 'Nunavut',
 35: 'Ontario'}

In [14]:
ProvinceIds = list(ProvinceID_To_ProvinceName_Map.keys())

In [15]:
All_Cluster_Labels = []

In [19]:
for pID in tqdm(ProvinceIDs, desc='Processing Provinces', unit='ProvinceID'):

    variable = 'seasonal_3'
    Province = ProvinceID_To_ProvinceName_Map[pID]
    Variable = 'Quarterly Seasonality'

    Sample = Get_Sample(provinceid=pID)
    Sample_Pivoted = Pivot_Data(Sample=Sample, col=variable)
    data_3d, data_2d = Pre_Process(Sample_Pivoted=Sample_Pivoted)
    upper_k = len(Sample_Pivoted)
    df_Metrics, K = Get_Metrics_And_K(data_3d=data_3d, data_2d=data_2d, upper_k=upper_k)
    AnnualPlot = Plot_K_Metrics(df_Metrics=df_Metrics, K=K, Province=Province, Variable=Variable)
    df_Clusters = Generate_Cluster_Labels(data_3d=data_3d, Sample_Pivoted=Sample_Pivoted, ClusterType='Trend', K=11)
    df_Clusters['provinceid'] = pID
    df_Clusters['Trend Cluster'] = df_Clusters['Trend Cluster'] + 1
    All_Cluster_Labels.append(df_Clusters)

    AnnualPlot.write_image(f'figures/{Province}_{K}.svg')
    



Processing Provinces: 100%|██████████| 10/10 [01:38<00:00,  9.85s/ProvinceID]


In [120]:
df_All_Cluster_Lables = pd.concat(All_Cluster_Labels, axis=0)
df_All_Cluster_Lables.head(10)

,noc_groupingid,Trend Cluster,provinceid
0,1,3,10.0
1,2,6,10.0
2,3,1,10.0
3,4,7,10.0
4,5,2,10.0
5,6,3,10.0
6,7,1,10.0
7,8,5,10.0
8,9,10,10.0
9,10,3,10.0


In [121]:
df_PD = df.to_pandas()


In [122]:
df_PD = df_PD.merge(df_All_Cluster_Lables, on=['provinceid','noc_groupingid'], how='left')
df_PD = df_PD.dropna(subset=['Trend Cluster'])

In [123]:
df_PD

,observed,trend,resid,seasonal_3,seasonal_6,seasonal_12,ds,provinceid,noc_groupingid,Trend Cluster
0,7.0,7.378831,-0.433604,-0.109472,-0.171635,0.335879,2000-01-01,10.0,1,3
1,8.2,7.410238,0.542999,-0.063244,0.075139,0.234868,2000-02-01,10.0,1,3
2,7.1,7.440902,0.183164,0.135241,-0.264254,-0.395053,2000-03-01,10.0,1,3
3,7.6,7.470413,0.108099,-0.047459,0.179786,-0.110839,2000-04-01,10.0,1,3
4,6.9,7.497976,-0.678920,-0.077763,-0.156244,0.314951,2000-05-01,10.0,1,3
...,...,...,...,...,...,...,...,...,...,...
181381,15.9,16.568508,-0.454525,-0.237140,-0.991029,1.014186,2025-05-01,59.0,60,3
181382,19.8,17.148172,1.821578,0.557542,-0.774380,1.047088,2025-06-01,59.0,60,3
181383,17.5,17.745231,0.064598,-0.269077,-0.662542,0.621789,2025-07-01,59.0,60,3
181384,18.4,18.357159,-0.750631,-0.561815,1.249591,0.105696,2025-08-01,59.0,60,3


In [124]:
df_PD.to_parquet('Data/SeasonalDecomposedDataClustered.parquet',index=False)